# Phase 5: LLM prompting baseline (Hindi NER)

Owner: Ranjeet Kumar Yadav. Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 5.

Zero-shot and few-shot prompting of a **self-hosted open LLM** (loaded and run in this notebook, no external API) to tag
PER / ORG / LOC in Hindi. The model answers with a JSON array; `scripts/phase5_parser.py` maps it back onto the word
tokens as IOB2 (rules R1 to R9 are written at the top of that file and tested in `scripts/test_phase5_parser.py`).
Scoring uses the **shared** `src.evaluate` (seqeval entity-level micro-F1), the same as every other stage.

**Malformed output is an error, never dropped.** It is scored as all `O` for that sentence and counted.

## Two stages, one notebook

| `STAGE` | Data scored | Purpose | Logs to `results.csv`? |
| --- | --- | --- | --- |
| `"validation"` | a fixed sample of the Hindi **val** split | pick the model, tune the prompt, check parser failures, speed, VRAM | never |
| `"final"` | **all 864** Hindi **gold test** sentences | the frozen model, prompt and parser, scored once per mode | yes (2 rows) |

**Do not tune on gold test.** Gold is only read in the final stage. (Its sentence texts are also loaded in the
validation stage, only to keep them out of the few-shot pool; their labels are not used.)

## How to run (Kaggle)
1. Commit and push this notebook, `scripts/phase5_parser.py` and `scripts/test_phase5_parser.py` to
   `phase5-llm-prompting`, because the run clones that branch and records its commit.
2. Import the notebook on Kaggle with a **GPU T4** and **Internet** on, and attach the `GH_TOKEN` (read-only) Secret.
   Neither candidate model is gated, so no `HF_TOKEN` is needed.
3. **Debug:** `STAGE = "validation"`, `DEBUG = True`, **Save & Run All** (a few minutes). Checks cloning, both model
   loads, generation, parsing and the MuRIL comparison on 16 sentences. Scores are meaningless; keep nothing.
4. **Validation:** `DEBUG = False`, `STAGE = "validation"`, **Save & Run All**. Edit the prompt in section 3 and
   re-run until the validation sample is stable. Record the final prompt's `PROMPT_HASH`.
5. Set `FINAL_MODEL` to the model the selection rule picked, freeze the prompt, and **wait for the lead's go-ahead**
   (Phase 4 must be merged first; both phases write `results.csv`).
6. **Final:** `STAGE = "final"`, `DEBUG = False`, one clean top-to-bottom **Save & Run All**. Download
   `NaamRekha/results/results.csv` and `NaamRekha/results/phase5/` from the Output tab, plus the executed notebook.
   Commit them unchanged.
7. Check the gate locally: `uv run python scripts/phase5_gate.py`.

In [ ]:
# Pin versions (docs/04-compute.md). Colab/Kaggle ship their own torch; the rest are pinned for reproducibility.
%pip install -q transformers==5.22.1 accelerate==1.15.0 seqeval==1.2.2 bitsandbytes==0.50.2 sentencepiece==0.2.2

In [ ]:
import base64
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"
REPO_BRANCH = "phase5-llm-prompting"


def get_secret(name):
    # Kaggle Secret, then Colab Secret, then environment variable.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook. Check for a repo file, not
# just src/, because Kaggle has its own unrelated /kaggle/src folder.
if os.path.isfile("../src/data.py"):
    REPO_ROOT = os.path.abspath("..")
elif os.path.isfile("src/data.py"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("NaamRekha")
    if not os.path.isdir(REPO_ROOT):
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        token = get_secret("GH_TOKEN")
        if token:
            # Passed through the environment so it is never saved in .git/config
            # (Kaggle keeps that folder as output) or printed if the clone fails.
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            env.update(GIT_CONFIG_COUNT="1",
                       GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
                       check=True, env=env)
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts"))

hf_token = get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token        # optional: only avoids Hub rate limits

GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_ROOT, "rev-parse", "--short", "HEAD"], text=True).strip()

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution
from phase5_parser import format_entities, parse_llm_output, tags_to_entities

print("repo:", REPO_ROOT, "| commit:", GIT_COMMIT)
print("labels:", LABELS)

## Parser tests (no GPU)

The parser is the riskiest part of this phase, so its tests run first. If any fail, nothing below should be trusted.

In [ ]:
proc = subprocess.run([sys.executable, "-X", "utf8", os.path.join(REPO_ROOT, "scripts", "test_phase5_parser.py")],
                      capture_output=True, text=True, encoding="utf-8")
print(proc.stdout[-400:])
assert proc.returncode == 0, proc.stdout + proc.stderr

## Config

Every knob is in this one cell. Both candidates are ungated, so nothing needs a token.

**Candidates (at most two, both fit a free T4).** Checked on the Hub: license, gating, exact revision.

| Key | Model | Load | License | Why |
| --- | --- | --- | --- | --- |
| `qwen2.5-3b` | `Qwen/Qwen2.5-3B-Instruct` | fp16 | `qwen-research` (research use only) | small and fast; the earlier draft's choice |
| `qwen2.5-7b-4bit` | `Qwen/Qwen2.5-7B-Instruct` | 4-bit nf4 | Apache-2.0 | larger multilingual model; fits a T4 in 4-bit |

Airavata (Llama 2 license) and Aya Expanse 8B (CC-BY-NC) are gated, and Navarasa uses the Gemma terms, so they were
left out to keep the run reproducible without extra access approvals.

**Selection rule (written before any validation run).** Among candidates whose few-shot malformed rate is at most 5%,
take the highest few-shot validation F1; if several are within 0.01 F1 of the best, take the faster one.

In [ ]:
STAGE = "validation"     # "validation": model + prompt selection on val, never gold. "final": frozen config on gold.
DEBUG = True             # True: 16 sentences, nothing logged or kept. False: the real runs.

# Kaggle's dual-T4 sessions: keep one GPU so device placement and speed are predictable.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

SEED = 42
LANG = "hi"

CANDIDATES = {
    "qwen2.5-3b": dict(model_id="Qwen/Qwen2.5-3B-Instruct", revision="aa8e72537993ba99e69dfaafa59ed015b17504d1",
                       load_4bit=False, license="qwen-research"),
    "qwen2.5-7b-4bit": dict(model_id="Qwen/Qwen2.5-7B-Instruct", revision="a09a35458c702b33eeacc393d103063234e8bc28",
                            load_4bit=True, license="apache-2.0"),
}
FINAL_MODEL = None       # set to a CANDIDATES key after validation, before the final run

VAL_RESERVED = 150       # val sentences reserved for validation; never used as few-shot examples
VAL_SAMPLE_SIZE = 150    # how many of them to score (<= VAL_RESERVED; seeded, same every run)
VAL_SAMPLE_SEED = 42
TEST_SIZE = None         # None = all 864 gold sentences (talk to the lead before changing)

PROMPT_VERSION = "v1"
K_SHOT = 8               # in-context examples in few-shot mode (incl. N_EMPTY_SHOTS answered with [])
N_EMPTY_SHOTS = 1
SHOT_SEED = 42
BATCH_SIZE = 8
MAX_NEW_TOKENS = 320     # Devanagari costs several tokens per word; a truncated answer counts as malformed
MAX_MALFORMED_RATE = 0.05
SELECTION_TIE = 0.01

if DEBUG:
    VAL_SAMPLE_SIZE = 16
    assert STAGE == "validation", "debug never touches gold"
if STAGE == "final":
    assert FINAL_MODEL in CANDIDATES, "set FINAL_MODEL to the model chosen on validation"
LOG_RESULTS = STAGE == "final" and not DEBUG
RUN_KEYS = list(CANDIDATES) if STAGE == "validation" else [FINAL_MODEL]
print("stage:", STAGE, "| debug:", DEBUG, "| models:", RUN_KEYS, "| log results:", LOG_RESULTS)

In [ ]:
import random
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def _ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "n/a"

LIB_VERSIONS = (f"torch={torch.__version__}; transformers={_ver('transformers')}; accelerate={_ver('accelerate')}; "
                f"bitsandbytes={_ver('bitsandbytes')}; seqeval={_ver('seqeval')}")

def pick_storage() -> str:
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/naamrekha"
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/NaamRekha"
    except ImportError:
        return os.path.join(REPO_ROOT, "checkpoints")

STORAGE = os.path.join(pick_storage(), "phase5")
os.makedirs(STORAGE, exist_ok=True)
# Repo-side artifacts (raw outputs, per-sentence predictions, errors). Debug output stays out of results/.
ART_DIR = STORAGE if DEBUG else os.path.join(REPO_ROOT, "results", "phase5")
os.makedirs(ART_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(LIB_VERSIONS)
print("device:", device, "| GPU count:", torch.cuda.device_count(), "| storage:", STORAGE, "| artifacts:", ART_DIR)
if device == "cpu":
    print("WARNING: no GPU. Parser/data cells work, model cells need a GPU (use Kaggle).")

## Data

* **Validation stage:** a seeded sample of the Hindi **val** split (silver labels, like Phase 3's size selection).
* **Final stage:** the full Hindi **gold test** set (864 sentences), per [docs/03-conventions.md](../docs/03-conventions.md).
* **Few-shot examples** come from the Hindi **val** split only, from sentences outside the validation sample and
  with no text shared with the gold test set, and without literal escape junk. They are short (6 to 25 words) and entity-rich, cover PER, ORG and LOC,
  and include `N_EMPTY_SHOTS` sentence(s) with no entities answered `[]`. Their labels are silver, a limitation.

In [ ]:
def n_ents(tags):
    return sum(t.startswith("B-") for t in tags)

val_all = load_split(LANG, "val")
test_all = load_split(LANG, "test")
print(f"val: {len(val_all)} sentences | gold test: {len(test_all)} sentences")

# The reserved validation sentences are fixed by seed in BOTH stages, so the few-shot pool (and the shots) is the
# same in validation and final. A smaller VAL_SAMPLE_SIZE (debug) is a prefix of the same random draw.
assert VAL_SAMPLE_SIZE <= VAL_RESERVED
reserved = random.Random(VAL_SAMPLE_SEED).sample(range(len(val_all)), VAL_RESERVED)

# Evaluation rows for this stage.
if STAGE == "validation":
    val_idx = sorted(reserved[:VAL_SAMPLE_SIZE])
    eval_rows = [val_all[i] for i in val_idx]
    eval_name = f"val sample {len(eval_rows)}/{len(val_all)} (seed {VAL_SAMPLE_SEED})"
else:
    val_idx = []
    eval_rows = test_all if TEST_SIZE is None else random.Random(VAL_SAMPLE_SEED).sample(test_all, TEST_SIZE)
    eval_name = f"gold test {len(eval_rows)}/{len(test_all)}"
print("evaluating on:", eval_name)
d = tag_distribution(eval_rows); tot = sum(d.values())
print({k: f"{v} ({v / tot:.1%})" for k, v in sorted(d.items())})

# Few-shot pool: val only, outside the reserved validation sentences and with no text shared with gold test.
fixed_val_idx = set(reserved)
test_texts = {tuple(w) for w, _ in test_all}
pool = [(i, r) for i, r in enumerate(val_all)
        if i not in fixed_val_idx and tuple(r[0]) not in test_texts and 6 <= len(r[0]) <= 25
        and not any("\\" in x for x in r[0])]        # skip rows with literal escape junk such as "\xa0"
rich = [r for _, r in pool if n_ents(r[1]) >= 2]
empty = [r for _, r in pool if n_ents(r[1]) == 0]
print(f"few-shot pool: {len(rich)} entity-rich, {len(empty)} entity-free")

def pick_shots(k, n_empty, seed):
    for attempt in range(1000):                                  # deterministic: first seed that covers all types
        r = random.Random(seed + attempt)
        chosen = r.sample(rich, k - n_empty)
        if {t[2:] for _, tags in chosen for t in tags if t.startswith("B-")} == {"PER", "ORG", "LOC"}:
            shots = chosen + r.sample(empty, n_empty)
            r.shuffle(shots)
            return shots
    raise RuntimeError("no shot set covers PER, ORG and LOC")

shots = pick_shots(K_SHOT, N_EMPTY_SHOTS, SHOT_SEED)
eval_texts = {tuple(w) for w, _ in eval_rows}
assert not any(tuple(w) in test_texts for w, _ in shots), "few-shot example overlaps the gold test set"
assert not any(tuple(w) in eval_texts for w, _ in shots), "few-shot example overlaps the evaluation sentences"
for w, t in shots:
    print("  shot:", " ".join(w), "->", format_entities(tags_to_entities(w, t)))

## Prompt

One fixed answer format, shared by both modes: `[{"text": "<entity as written>", "type": "PER|ORG|LOC"}]`.
Zero-shot is the instructions alone; few-shot is the same instructions plus the fixed shots above as chat turns.
`PROMPT_HASH` fingerprints the instructions and the shots so a run can be matched to the prompt that produced it.
**Edit the prompt only in the validation stage**, then freeze it.

In [ ]:
import hashlib

SYSTEM = (
    "You are a Named Entity Recognition system for Hindi text. "
    "Find every person (PER), organisation (ORG) and location (LOC) name in the sentence. "
    "PER: names of people. ORG: companies, political parties, institutions, government bodies, teams. "
    "LOC: countries, states, cities, rivers and other places. "
    "Copy each entity exactly as it is written in the sentence: same script, same spelling, no translation. "
    "Reply with ONLY a JSON array and no explanation, in this format: "
    '[{"text": "<entity as written>", "type": "PER"}, {"text": "<entity as written>", "type": "LOC"}]. '
    "Use the type PER, ORG or LOC. If the sentence has no entities, reply []."
)

def user_msg(words):
    return "Sentence: " + " ".join(words)

def build_messages(words, shots=()):
    msgs = [{"role": "system", "content": SYSTEM}]
    for w, t in shots:
        msgs += [{"role": "user", "content": user_msg(w)},
                 {"role": "assistant", "content": format_entities(tags_to_entities(w, t))}]
    msgs.append({"role": "user", "content": user_msg(words)})
    return msgs

PARSER_SHA = hashlib.sha1(open(os.path.join(REPO_ROOT, "scripts", "phase5_parser.py"), "rb").read()).hexdigest()[:8]
PROMPT_HASH = hashlib.sha1((PROMPT_VERSION + SYSTEM + json.dumps(build_messages([""], shots), ensure_ascii=False))
                           .encode("utf-8")).hexdigest()[:8]
print(f"prompt {PROMPT_VERSION}/{PROMPT_HASH} | parser {PARSER_SHA}")
print(json.dumps(build_messages(eval_rows[0][0], shots[:1]), ensure_ascii=False, indent=1)[:1500])

In [ ]:
# Parser ceiling: feed the GOLD answers through the parser. Anything below 1.0 is lost by the parser rules
# (e.g. a name tagged once in gold but mentioned twice), not by the model.
ceiling_pred = [parse_llm_output(format_entities(tags_to_entities(w, t)), w).tags for w, t in eval_rows]
CEILING = evaluate([t for _, t in eval_rows], ceiling_pred)
print("parser ceiling on", eval_name, ":", {k: round(v, 4) for k, v in CEILING.items()})

## Model loading, generation and scoring

Each candidate is loaded from its **pinned revision**, run in both modes, then freed (`del` +
`torch.cuda.empty_cache()`) before the next model, so only one large model is in VRAM at a time.
Greedy decoding. Raw outputs are cached under `STORAGE` so a disconnect does not lose generated text.

In [ ]:
import gc
import time
from collections import Counter
from tqdm.auto import tqdm

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_llm(cfg):
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU attached. Kaggle: Settings > Accelerator > GPU T4. Colab: Runtime > T4 GPU.")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(cfg["model_id"], revision=cfg["revision"], padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(revision=cfg["revision"], device_map={"": 0})
    if cfg["load_4bit"]:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16)
    else:
        kw["dtype"] = torch.float16
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], **kw).eval()
    return tok, model

@torch.no_grad()
def generate(tok, model, rows, shots, cache_path):
    """Greedy answers for every row. Returns (raw texts, truncated flags, seconds spent generating)."""
    if os.path.exists(cache_path):
        cached = [json.loads(l) for l in open(cache_path, encoding="utf-8")]
        if len(cached) == len(rows):
            print(f"  loaded {len(cached)} cached outputs from {cache_path}")
            return [c["raw"] for c in cached], [c["truncated"] for c in cached], None
    stop = set(np.atleast_1d(model.generation_config.eos_token_id).tolist()) | {tok.pad_token_id}
    order = sorted(range(len(rows)), key=lambda i: len(rows[i][0]))        # similar lengths per batch = less padding
    raw, trunc = [None] * len(rows), [False] * len(rows)
    t0 = time.time()
    for b in tqdm(range(0, len(order), BATCH_SIZE), desc="generate"):
        ids = order[b:b + BATCH_SIZE]
        prompts = [tok.apply_chat_template(build_messages(rows[i][0], shots), tokenize=False, add_generation_prompt=True)
                   for i in ids]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        for i, row, text in zip(ids, gen.tolist(), tok.batch_decode(gen, skip_special_tokens=True)):
            raw[i], trunc[i] = text, not any(t in stop for t in row)       # no stop token = ran out of tokens
    secs = time.time() - t0
    with open(cache_path, "w", encoding="utf-8") as f:
        for r, t in zip(raw, trunc):
            f.write(json.dumps({"raw": r, "truncated": t}, ensure_ascii=False) + "\n")
    return raw, trunc, secs

def score(rows, raw, truncated):
    parsed = [parse_llm_output(r, w) for r, (w, _) in zip(raw, rows)]
    gold = [t for _, t in rows]
    scores = evaluate(gold, [p.tags for p in parsed])
    stats = dict(
        n=len(rows),
        malformed=sum(p.malformed for p in parsed),
        malformed_reasons=dict(Counter(p.reason for p in parsed if p.malformed)),
        truncated=sum(truncated),
        noisy=sum(p.noisy for p in parsed),
        invalid_items=sum(p.n_invalid_items for p in parsed),
        unknown_label=sum(p.n_unknown_label for p in parsed),
        unaligned=sum(p.n_unaligned for p in parsed),
        overlap_dropped=sum(p.n_overlap_dropped for p in parsed),
        conflict=sum(p.n_conflict for p in parsed),
        items=sum(p.n_items for p in parsed),
    )
    return scores, stats, parsed

In [ ]:
results = {}     # (model key, mode) -> dict
MODES = [("zero-shot", ()), ("few-shot", tuple(shots))]

for key in RUN_KEYS:
    cfg = CANDIDATES[key]
    print(f"\n=== {key}: {cfg['model_id']} @ {cfg['revision'][:8]} | 4bit={cfg['load_4bit']} | license={cfg['license']}")
    free_gpu()
    tok, model = load_llm(cfg)
    fingerprint = hashlib.sha1(json.dumps([cfg["model_id"], cfg["revision"], cfg["load_4bit"], PROMPT_HASH, PARSER_SHA,
                                           K_SHOT, N_EMPTY_SHOTS, SHOT_SEED, BATCH_SIZE, MAX_NEW_TOKENS]).encode()
                               ).hexdigest()[:8]
    print("loaded | VRAM after load (GB):", round(torch.cuda.memory_allocated() / 1e9, 2), "| config fingerprint:", fingerprint)
    for mode, sh in MODES:
        torch.cuda.reset_peak_memory_stats()
        cache = os.path.join(STORAGE, f"{STAGE}_{key}_{mode}_{PROMPT_HASH}_{len(eval_rows)}.jsonl")
        raw, trunc, secs = generate(tok, model, eval_rows, sh, cache)
        scores, stats, parsed = score(eval_rows, raw, trunc)
        results[(key, mode)] = dict(cfg=cfg, scores=scores, stats=stats, parsed=parsed, raw=raw, truncated=trunc,
                                    sec_per_sent=None if secs is None else secs / len(eval_rows),
                                    peak_gb=torch.cuda.max_memory_allocated() / 1e9, fingerprint=fingerprint)
        print(f"[{key} / {mode}] F1 overall {scores['F1_overall']:.4f} | PER {scores['F1_PER']:.4f} | "
              f"ORG {scores['F1_ORG']:.4f} | LOC {scores['F1_LOC']:.4f}")
        print(f"   malformed {stats['malformed']}/{stats['n']} {stats['malformed_reasons']} | truncated {stats['truncated']} | "
              f"unaligned spans {stats['unaligned']} | peak VRAM {results[(key, mode)]['peak_gb']:.1f} GB")
    del model, tok
    free_gpu()

## Summary and model selection

Speed and VRAM matter for the final run: 864 sentences in each of two modes has to fit one Kaggle session.

In [ ]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

rows_ = []
for (key, mode), r in results.items():
    s, st = r["scores"], r["stats"]
    sec = r["sec_per_sent"]
    rows_.append({"model": key, "mode": mode, "F1": round(s["F1_overall"], 4), "PER": round(s["F1_PER"], 4),
                  "ORG": round(s["F1_ORG"], 4), "LOC": round(s["F1_LOC"], 4),
                  "malformed": f"{st['malformed']}/{st['n']}", "truncated": st["truncated"], "unaligned": st["unaligned"],
                  "unknown_label": st["unknown_label"], "sec/sent": None if sec is None else round(sec, 2),
                  "est. min for 864": None if sec is None else round(sec * 864 / 60, 1),
                  "peak VRAM GB": round(r["peak_gb"], 1)})
summary = pd.DataFrame(rows_)
display(summary)
print(f"parser ceiling {CEILING['F1_overall']:.4f} | evaluated on {eval_name}")

def pick_model():
    few = {k: r for (k, m), r in results.items() if m == "few-shot" and r["stats"]["malformed"] / r["stats"]["n"] <= MAX_MALFORMED_RATE}
    if not few:
        return None
    best = max(r["scores"]["F1_overall"] for r in few.values())
    close = [k for k, r in few.items() if r["scores"]["F1_overall"] >= best - SELECTION_TIE]
    return min(close, key=lambda k: results[(k, "few-shot")]["sec_per_sent"] or 0)

if STAGE == "validation":
    chosen = pick_model()
    print("selection rule picks:", chosen, "(set FINAL_MODEL to this, then freeze the prompt)" if chosen
          else "-> no candidate met the malformed-output limit; fix the prompt first")
    if DEBUG:
        print("DEBUG run: numbers above only prove the pipeline works.")

## Log to the shared results sheet (final stage only)

One row per mode. `checkpoint_path` is the model's pinned Hugging Face URL. The row is printed exactly as written so
the gate can match it to this notebook's output.

In [ ]:
def notes_for(key, mode, r):
    st, cfg = r["stats"], r["cfg"]
    return (f"phase=5; mode={mode}; k_shot={K_SHOT if mode == 'few-shot' else 0}; test_n={st['n']}/{len(test_all)}; "
            f"sample_seed=n/a (full test); seed={SEED}; shot_seed={SHOT_SEED}; model={cfg['model_id']}; "
            f"revision={cfg['revision']}; load_4bit={cfg['load_4bit']}; license={cfg['license']}; greedy; "
            f"batch={BATCH_SIZE}; max_new_tokens={MAX_NEW_TOKENS}; prompt={PROMPT_VERSION}/{PROMPT_HASH}; "
            f"parser={PARSER_SHA}; config={r['fingerprint']}; malformed={st['malformed']}; "
            f"malformed_rate={st['malformed'] / st['n']:.4f}; truncated={st['truncated']}; noisy={st['noisy']}; "
            f"invalid_items={st['invalid_items']}; unknown_label={st['unknown_label']}; unaligned_spans={st['unaligned']}; "
            f"git={GIT_COMMIT}; {LIB_VERSIONS}")

if LOG_RESULTS:
    assert len(RUN_KEYS) == 1 and len(eval_rows) == len(test_all), "final run = one model, all gold sentences"
    from src import RESULTS_PATH
    for (key, mode), r in results.items():
        cfg = r["cfg"]
        log_result(model=f"LLM-{mode} ({cfg['model_id'].split('/')[-1]})", train_lang="none", test_lang=LANG,
                   subset_size=0, epochs=0, scores=r["scores"],
                   checkpoint_path=f"https://huggingface.co/{cfg['model_id']}/tree/{cfg['revision']}",
                   notes=notes_for(key, mode, r))
    with open(RESULTS_PATH, encoding="utf-8") as f:
        for line in f:
            if line.startswith("LLM-"):
                print(line.rstrip("\n"))                  # exactly the logged rows
else:
    print("not the final stage (or DEBUG): nothing written to results.csv")

## MuRIL comparison model (Phase 3 handoff checkpoint)

The public handoff model at its pinned revision, run on the **same sentences** as the LLM so errors can be compared.
The LLM is already freed. On the full gold test set this must reproduce Phase 3's logged score (**0.8274**), which
checks this inference code.

In [ ]:
MURIL_REPO = "priyadarshi-su/naamrekha-muril_hi_n300000_e3_lr2e-05_s42_bfd254ed"
MURIL_REV = "de5204680ed4a8ab1ac842847df6866a6e2c9251"

from transformers import AutoModelForTokenClassification, AutoTokenizer
free_gpu()
m_tok = AutoTokenizer.from_pretrained(MURIL_REPO, revision=MURIL_REV)
m_model = AutoModelForTokenClassification.from_pretrained(MURIL_REPO, revision=MURIL_REV).to(device).eval()
cfg_labels = {int(k): v for k, v in m_model.config.id2label.items()}
assert cfg_labels == id2label, f"checkpoint label order differs from the shared label map: {cfg_labels}"

@torch.no_grad()
def predict_muril(rows, max_length=128, bs=32):
    preds = []
    for b in tqdm(range(0, len(rows), bs), desc="muril"):
        batch = [w for w, _ in rows[b:b + bs]]
        enc = m_tok(batch, is_split_into_words=True, truncation=True, max_length=max_length, padding=True, return_tensors="pt")
        ids = m_model(**{k: v.to(device) for k, v in enc.items()}).logits.argmax(-1).cpu().tolist()
        for j, words in enumerate(batch):
            tags, seen = ["O"] * len(words), set()
            for pos, wid in enumerate(enc.word_ids(j)):
                if wid is not None and wid not in seen:          # first subword carries the word's label
                    seen.add(wid)
                    tags[wid] = id2label[ids[j][pos]]
            preds.append(tags)                                   # words lost to truncation stay O
    return preds

muril_pred = predict_muril(eval_rows)
muril_scores = evaluate([t for _, t in eval_rows], muril_pred)
print("MuRIL on", eval_name, {k: round(v, 4) for k, v in muril_scores.items()})
if STAGE == "final" and len(eval_rows) == len(test_all):
    assert abs(muril_scores["F1_overall"] - 0.8274) < 0.002, "MuRIL does not reproduce Phase 3's 0.8274: check inference code"
    print("reproduces the Phase 3 handoff score (0.8274)")
del m_model, m_tok
free_gpu()

## Error analysis

Entity-level errors per system, compared with the gold spans. Every example shows **both** the gold span and what the
system predicted.

| Kind | Meaning |
| --- | --- |
| `missed` | a gold entity with no predicted span touching it |
| `boundary` | a predicted span overlaps the gold entity but with different boundaries (type may differ) |
| `type_confusion` | the exact gold span was predicted with another type |
| `spurious` | a predicted span that touches no gold entity |
| `format_drift` | malformed output (listed separately, not an entity error) |

In [ ]:
from seqeval.metrics.sequence_labeling import get_entities

def spans(tags):
    return [(s, e, t) for t, s, e in get_entities(tags)]

def classify(words, gold, pred):
    G, P = spans(gold), spans(pred)
    gset, pset = set(G), set(P)
    text = lambda sp: " ".join(words[sp[0]:sp[1] + 1])
    out = []
    for g in G:
        if g in pset:
            continue
        same = [p for p in P if (p[0], p[1]) == (g[0], g[1])]
        over = [p for p in P if (p[0], p[1]) != (g[0], g[1]) and not (p[1] < g[0] or p[0] > g[1])]
        kind, p = ("type_confusion", same[0]) if same else ("boundary", over[0]) if over else ("missed", None)
        out.append(dict(kind=kind, gold=text(g), gold_type=g[2], pred=text(p) if p else "", pred_type=p[2] if p else ""))
    for p in P:
        if p not in gset and not any(not (g[1] < p[0] or g[0] > p[1]) for g in G):
            out.append(dict(kind="spurious", gold="", gold_type="", pred=text(p), pred_type=p[2]))
    return out

systems = {f"LLM {mode} [{key}]": r["parsed"] for (key, mode), r in results.items()}
preds = {name: [p.tags for p in parsed] for name, parsed in systems.items()}
preds["MuRIL (Phase 3)"] = muril_pred

err_rows = []
for name, ptags in preds.items():
    for i, ((w, g), p) in enumerate(zip(eval_rows, ptags)):
        for e in classify(w, g, p):
            err_rows.append(dict(system=name, idx=i, sentence=" ".join(w), **e))
err = pd.DataFrame(err_rows)
print("error counts by kind:")
display(err.groupby(["system", "kind"]).size().unstack(fill_value=0))
print("by gold/predicted type (type = gold type, or predicted type for spurious):")
err["type"] = err.gold_type.where(err.gold_type != "", err.pred_type)
display(err.groupby(["system", "type", "kind"]).size().unstack(fill_value=0))

In [ ]:
def show(system, mask, n=10, cols=("kind", "gold_type", "gold", "pred_type", "pred", "sentence")):
    sub = err[(err.system == system) & mask].head(n)
    display(sub[list(cols)].assign(sentence=sub.sentence.str.slice(0, 90)))

_few = {f"LLM {m} [{k}]": r["scores"]["F1_overall"] for (k, m), r in results.items() if m == "few-shot"}
best_llm = max(_few, key=_few.get)          # the few-shot run with the highest F1 on these sentences
print("=== ORG errors (the weakest type across models):", best_llm)
show(best_llm, (err["type"] == "ORG"), n=12)
print("=== ORG errors: MuRIL")
show("MuRIL (Phase 3)", (err["type"] == "ORG"), n=12)
for kind in ["boundary", "missed", "spurious", "type_confusion"]:
    print(f"\n=== {kind}: {best_llm}")
    show(best_llm, (err.kind == kind), n=8)

In [ ]:
# Format drift and unaligned spans, per LLM run.
for (key, mode), r in results.items():
    st = r["stats"]
    print(f"\n=== {key} / {mode}: malformed {st['malformed']}/{st['n']} {st['malformed_reasons']} | "
          f"truncated {st['truncated']} | with prose or fence {st['noisy']} | invalid items {st['invalid_items']} | "
          f"unknown labels {st['unknown_label']} | unaligned spans {st['unaligned']} | overlap-dropped {st['overlap_dropped']}")
    bad = [i for i, p in enumerate(r["parsed"]) if p.malformed]
    for i in bad[:6]:
        print(f"  malformed [{r['parsed'][i].reason}] ->", repr(r["raw"][i][:160]))
    un = Counter(t for p in r["parsed"] for t in p.unaligned_texts)
    print("  most common unaligned spans (text the model wrote that is not in the sentence):", un.most_common(10))

# Sentences where MuRIL is exactly right and the LLM is not, and the reverse.
g_all = [t for _, t in eval_rows]
muril_ok = [p == g for p, g in zip(muril_pred, g_all)]
for name in [s for s in preds if s.startswith("LLM")]:
    llm_ok = [p == g for p, g in zip(preds[name], g_all)]
    print(f"{name}: both exact {sum(a and b for a, b in zip(muril_ok, llm_ok))} | MuRIL only {sum(a and not b for a, b in zip(muril_ok, llm_ok))} "
          f"| LLM only {sum(b and not a for a, b in zip(muril_ok, llm_ok))} | neither {sum(not a and not b for a, b in zip(muril_ok, llm_ok))}")

## Save per-sentence artifacts

Raw model output, parse flags and predicted tags for every sentence and system, plus the error table. These back every
claim in `docs/phase5/` (final stage: download them from the Output tab and commit unchanged).

In [ ]:
pred_path = os.path.join(ART_DIR, f"{STAGE}_predictions.jsonl")
with open(pred_path, "w", encoding="utf-8") as f:
    for name, ptags in preds.items():
        llm = next(((k, m) for (k, m) in results if f"LLM {m} [{k}]" == name), None)
        for i, ((w, g), p) in enumerate(zip(eval_rows, ptags)):
            rec = dict(system=name, idx=i, words=w, gold=g, pred=p)
            if llm:
                pr = results[llm]["parsed"][i]
                rec.update(raw=results[llm]["raw"][i], malformed=pr.malformed, reason=pr.reason,
                           unaligned=pr.unaligned_texts, truncated=results[llm]["truncated"][i])
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
err_path = os.path.join(ART_DIR, f"{STAGE}_errors.csv")
err.to_csv(err_path, index=False, encoding="utf-8")
print("wrote", pred_path, "and", err_path)

## Comparison with the other stages (from `results.csv`)

Read from the committed results sheet, not retyped. Phase 4 rows are not used.

In [ ]:
import csv
from src import RESULTS_PATH
with open(RESULTS_PATH, encoding="utf-8", newline="") as f:
    sheet = list(csv.DictReader(f))
want = [r for r in sheet if r["test_lang"] == "hi" and (r["model"].startswith("LLM-")
        or (r["model"] == "BiLSTM-CRF") or (r["model"] in ("MuRIL", "IndicBERT") and r["subset_size"] == "300000" and "s42" in r["checkpoint_path"]))]
display(pd.DataFrame([{k: (round(float(r[k]), 4) if k.startswith("F1") else r[k]) for k in ("model", "subset_size", "F1_PER", "F1_ORG", "F1_LOC", "F1_overall")} for r in want]))

## Findings

Written after the run, from the printed outputs above. Leave this empty until then; the final write-up lives in
`docs/phase5/README.md` and `docs/phase5/error_analysis.md`.